# Lunar Polar Temperature Table (LRO Diviner)

**What this notebook does, in plain words**

1. Downloads NASA's Diviner temperature maps of the Moon's polar regions (public, no account needed).
2. Keeps only a small circle around the site you pick (default: Shackleton crater, south pole).
3. Gives you a **table of surface temperatures (in kelvin)** for each of the 96 steps of a lunar day,
   for summer and for winter, and saves it as **CSV and Excel**.

**Important: this is not a date-by-date history.** The Diviner polar maps are *averages* of every
observation made between July 2009 and February 2019, grouped by where the Sun was in the sky. So the table
answers "how hot or cold is this spot at each moment of a typical lunar day", not "what was the temperature
on 3 March 2015". One lunar day lasts about 29.5 Earth days, so each of the 96 steps is about 7.4 hours.

**How to use it:** run every cell from top to bottom. The first run is a *quick test* (3 of the 96 steps) to
make sure the download works. Then set `QUICK_TEST = False` in Step 1 and run again for the full table.

## Step 1 — Choose the site

Pick a named site or enter your own latitude/longitude. Longitude is degrees **east**, 0 to 360. The site
coordinates below are approximate; check them against the IAU Gazetteer if precision matters.

In [ ]:
SITES = {                       # (latitude, east longitude) - approximate
    'Shackleton': (-89.9, 129.8),
    'Haworth':    (-87.5, 355.0),
    'Shoemaker':  (-88.1,  44.9),
    'Faustini':   (-87.3,  77.0),
    'Cabeus':     (-84.9, 324.5),
    'Peary':      ( 88.6,  33.0),   # north pole
}
SITE = 'Shackleton'             # or None to use the custom coordinates below
CUSTOM_LAT, CUSTOM_LON = -89.0, 0.0

RADIUS_KM = 15                  # size of the circle around the site
QUICK_TEST = True               # True = only 3 of 96 steps (fast check). False = full table.

if SITE:
    LAT, LON = SITES[SITE]
else:
    LAT, LON = CUSTOM_LAT, CUSTOM_LON
POLE = 'POLS' if LAT < 0 else 'POLN'            # south or north pole, from the latitude sign
SEASONS = ['SUM', 'WIN']                        # summer / winter (defined by where the Sun is relative to the equator)
N_BINS = 96
BINS = [1, 2, 3] if QUICK_TEST else list(range(1, N_BINS + 1))
DEG_PER_BIN = 360 / N_BINS                      # 3.75 degrees of subsolar longitude per step
HOURS_PER_BIN = 29.53 * 24 / N_BINS             # about 7.4 hours per step
print(f'Site: {SITE or "custom"} ({LAT}, {LON}) | radius {RADIUS_KM} km | pole {POLE} | steps: {len(BINS)} of {N_BINS}')

## Step 2 — Download tools (no need to read this cell)

These functions find the Diviner files on the NASA PDS server, read them in chunks (each file covers the
whole 10-degree polar cap and is large), and keep only the points inside your circle.

In [ ]:
import io, os, re, time, math
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

R_MOON_KM = 1737.4
PDS_BASE = ('https://pds-geosciences.wustl.edu/lro/urn-nasa-pds-lro_diviner_derived1/'
            'data_derived_pcp/diurnal/slon/')
HEADERS = {'User-Agent': 'lunar-temperature-table/1.0 (research notebook)'}
FILE_RE = re.compile(r'PCP_AVG_TBOL_(POL[NS])_(SUM|WIN)_SLON0*(\d+)_240\.TAB$', re.I)

def _list_dir(url):
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return [urljoin(url, h) for h in re.findall(r'href="([^"]+)"', r.text, flags=re.I)]

def index_pcp_files(pole, base=None, max_depth=3):
    """Walk the server folders and return {(season, step): url} for one pole."""
    base = base or PDS_BASE
    found, visited = {}, set()
    todo = [(base + pole.lower() + '/', 0)]
    while todo:
        url, depth = todo.pop()
        if url in visited or depth > max_depth:
            continue
        visited.add(url)
        for link in _list_dir(url):
            if not link.startswith(url) or link == url:     # skip "parent directory" links
                continue
            if link.endswith('/'):
                todo.append((link, depth + 1))
                continue
            m = FILE_RE.search(link.split('?')[0])
            if m and m.group(1).upper() == pole.upper():
                found[(m.group(2).upper(), int(m.group(3)))] = link
    return found

def _distance_km(lat, lon, lat0, lon0):
    la, lo = np.radians(lat), np.radians(lon)
    la0, lo0 = math.radians(lat0), math.radians(lon0)
    a = np.sin((la - la0) / 2) ** 2 + np.cos(la) * math.cos(la0) * np.sin((lo - lo0) / 2) ** 2
    return 2 * R_MOON_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def _normalize_columns(df):
    rename = {}
    for c in df.columns:
        cl = str(c).lower()
        if 'tbol' in cl or cl.startswith('tb') or 'temp' in cl: rename[c] = 'tbol'
        elif 'lat' in cl: rename[c] = 'lat'
        elif 'lon' in cl: rename[c] = 'lon'
        elif cl.startswith('x'): rename[c] = 'x'
        elif cl.startswith('y'): rename[c] = 'y'
    df = df.rename(columns=rename)
    missing = {'tbol', 'lat', 'lon'} - set(df.columns)
    if missing:
        raise ValueError(f'Could not recognise columns {missing} in the PCP file (found {list(df.columns)}).')
    return df

def read_file_roi(url, lat0, lon0, radius_km, chunk_mb=64, timeout=180):
    """Stream one PCP file and keep only the pixels inside the circle."""
    state = {'cols': None, 'sep': None}
    parts = []
    dlat = radius_km / (R_MOON_KM * math.pi / 180) * 1.05

    def _process(block):
        if state['sep'] is None:
            first = block.split(b'\n', 1)[0].decode('ascii', 'ignore').strip()
            state['sep'] = ',' if ',' in first else r'\s+'
            if re.search('[A-Za-z]', first):                        # the file has a header line
                state['cols'] = [c.strip().strip('"').lower() for c in re.split(state['sep'], first) if c.strip()]
                block = block.split(b'\n', 1)[1] if b'\n' in block else b''
            else:                                                   # no header: documented order
                state['cols'] = ['x', 'y', 'lon', 'lat', 'tbol']
        if not block.strip():
            return
        df = pd.read_csv(io.BytesIO(block), sep=state['sep'], header=None, names=state['cols'],
                         comment='#', skipinitialspace=True)
        df = _normalize_columns(df)
        df['lon'] = df['lon'] % 360
        df = df[(df['lat'] - lat0).abs() <= dlat]
        if len(df):
            d = _distance_km(df['lat'].values, df['lon'].values, lat0, lon0)
            parts.append(df[d <= radius_km])

    with requests.get(url, headers=HEADERS, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        rest = b''
        for chunk in r.iter_content(chunk_size=int(chunk_mb * 1024 * 1024)):
            data = rest + chunk
            cut = data.rfind(b'\n')
            if cut < 0:
                rest = data
                continue
            _process(data[:cut + 1])
            rest = data[cut + 1:]
        if rest.strip():
            _process(rest)
    cols = ['x', 'y', 'lon', 'lat', 'tbol']
    if not parts:
        return pd.DataFrame(columns=cols)
    out = pd.concat(parts, ignore_index=True)
    return out[[c for c in cols if c in out.columns]]

def estimate_volume(index, seasons, bins, n_samples=3):
    """Ask the server for the size of a few files (downloads nothing) and estimate the total."""
    keys = [(s, b) for s in seasons for b in bins if (s, b) in index]
    sizes = []
    for k in keys[:n_samples]:
        try:
            h = requests.head(index[k], headers=HEADERS, timeout=30, allow_redirects=True)
            if h.headers.get('Content-Length'):
                sizes.append(int(h.headers['Content-Length']))
        except Exception:
            pass
    if sizes:
        print(f'Files to read: {len(keys)} | average sampled size: {np.mean(sizes)/1e6:.0f} MB '
              f'| estimated total: ~{np.mean(sizes) * len(keys) / 1e9:.1f} GB')
    else:
        print(f'Files to read: {len(keys)} | the server did not report file sizes.')

def download_roi(index, pole, seasons, bins, lat0, lon0, radius_km, cache_dir='diviner_cache',
                 workers=4, chunk_mb=64):
    os.makedirs(cache_dir, exist_ok=True)
    missing = [(s, b) for s in seasons for b in bins if (s, b) not in index]
    if missing:
        raise RuntimeError(f'{len(missing)} expected files were not found on the server, e.g. {missing[:4]}')
    tag = f'{lat0:.2f}_{lon0:.2f}_{radius_km:.0f}km'

    def _job(season, b):
        path = os.path.join(cache_dir, f'{pole}_{season}_SLON{b:02d}_{tag}.csv.gz')
        if os.path.exists(path):                      # already downloaded before
            return season, b, pd.read_csv(path)
        last = None
        for attempt in range(3):
            try:
                df = read_file_roi(index[(season, b)], lat0, lon0, radius_km, chunk_mb)
                break
            except Exception as e:
                last = e
                time.sleep(5 * (attempt + 1))
        else:
            raise RuntimeError(f'Download failed for {season} step {b}: {last}')
        df.to_csv(path, index=False)
        return season, b, df

    results, t0 = [], time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = [ex.submit(_job, s, b) for s in seasons for b in bins]
        for i, f in enumerate(as_completed(futures), 1):
            season, b, df = f.result()
            results.append(df.assign(season=season, step=b))
            if i % 6 == 0 or i == len(futures):
                print(f'  {i}/{len(futures)} files done ({time.time() - t0:.0f} s)')
    return pd.concat(results, ignore_index=True)
print('Tools ready.')

## Step 3 — Download the data for your circle

First it lists the files on the server and estimates the download size, then it downloads. Files already
downloaded are kept in the `diviner_cache` folder, so if the connection drops you can run this cell again and
it continues where it stopped. In Colab that folder is erased when the session ends.

In [ ]:
print('Looking up files on the NASA PDS server...')
index = index_pcp_files(POLE)
print(f'{len(index)} files found for {POLE}.')
estimate_volume(index, SEASONS, BINS)

raw = download_roi(index, POLE, SEASONS, BINS, LAT, LON, RADIUS_KM)
raw = raw[(raw['tbol'] >= 10) & (raw['tbol'] <= 450)]        # drop obviously invalid temperatures
if raw.empty:
    raise RuntimeError('No pixels fell inside the circle. Check the latitude/longitude or raise RADIUS_KM.')
print(f'Pixel readings kept inside the circle: {len(raw):,}')

## Step 4 — Build the table

For each season and each of the 96 steps of the lunar day, the table gives the **mean, minimum and maximum
temperature (K)** over all the pixels inside your circle, and how many pixels there were. "Subsolar
longitude" is where the Sun is directly overhead; it moves 360 degrees over one lunar day.

In [ ]:
table = (raw.groupby(['season', 'step'])['tbol']
            .agg(n_pixels='count', temp_mean_K='mean', temp_min_K='min', temp_max_K='max')
            .reset_index())
table['subsolar_lon_start_deg'] = (table['step'] - 1) * DEG_PER_BIN
table['subsolar_lon_end_deg'] = table['step'] * DEG_PER_BIN
table['approx_hour_in_lunar_day'] = (table['step'] - 1) * HOURS_PER_BIN
table = table[['season', 'step', 'subsolar_lon_start_deg', 'subsolar_lon_end_deg',
               'approx_hour_in_lunar_day', 'n_pixels', 'temp_mean_K', 'temp_min_K', 'temp_max_K']].round(2)

summary = (table.groupby('season')
                .agg(coldest_K=('temp_min_K', 'min'), warmest_K=('temp_max_K', 'max'),
                     average_of_means_K=('temp_mean_K', 'mean')).round(2).reset_index())

name = f'lunar_temperature_{(SITE or "custom").lower()}' + ('_QUICKTEST' if QUICK_TEST else '')
table.to_csv(name + '.csv', index=False)
try:
    with pd.ExcelWriter(name + '.xlsx') as xw:
        table.to_excel(xw, sheet_name='temperature_table', index=False)
        summary.to_excel(xw, sheet_name='summary', index=False)
    saved = [name + '.csv', name + '.xlsx']
except Exception as e:                                           # e.g. no Excel writer installed
    print('Excel file not written (', e, '); CSV is saved.')
    saved = [name + '.csv']
print('Saved:', saved)
if QUICK_TEST:
    print(f'QUICK TEST: only {len(BINS)} of {N_BINS} steps. Set QUICK_TEST = False in Step 1 for the full table.')

try:
    display(summary); display(table.head(20))
except NameError:
    print(summary.to_string(index=False)); print(table.head(20).to_string(index=False))

## Step 5 — Plot and download the files

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
colors = {'SUM': '#E85D04', 'WIN': '#0077B6'}
labels = {'SUM': 'summer', 'WIN': 'winter'}
for s in SEASONS:
    d = table[table['season'] == s].sort_values('step')
    ax.plot(d['step'], d['temp_mean_K'], color=colors[s], marker='o', ms=3, label=f'{labels[s]} - mean')
    ax.fill_between(d['step'], d['temp_min_K'], d['temp_max_K'], color=colors[s], alpha=0.2,
                    label=f'{labels[s]} - min to max in circle')
ax.axhline(110, color='gray', linestyle='--', linewidth=1, label='110 K (reference for water-ice stability)')
ax.set_xlabel(f'Step of the lunar day (1-{N_BINS}, about {HOURS_PER_BIN:.1f} h each)')
ax.set_ylabel('Temperature (K)')
ax.set_title(f'{SITE or "Custom site"}: surface temperature across a typical lunar day (Diviner, 2009-2019 average)')
ax.grid(alpha=0.2); ax.legend()
plt.tight_layout(); plt.show()

try:                                       # in Google Colab this downloads the files to your computer
    from google.colab import files
    for f in saved:
        files.download(f)
except ImportError:
    pass

## Notes

- **Data source:** LRO Diviner Polar Cumulative Products (diurnal, by subsolar longitude), NASA PDS
  Geosciences Node. Temperature is the *bolometric brightness temperature* in kelvin at 240 m per pixel,
  covering the 10 degrees of latitude around each pole.
- **Min/max in the table** are the lowest and highest values among the *averaged* pixel values in your circle.
  Because each value is already an average of many passes, the true instantaneous extremes can be a bit more
  extreme than what the table shows.
- **Want a real date-by-date history?** That would have to come from Diviner's calibrated records (RDR). They
  are huge (tens of gigabytes per day), so it only makes sense for a short time window and a small area.
- **Reference temperature:** 110 K is only a commonly used rule of thumb for water-ice stability. Change or
  remove the line in Step 5 as you like.